# 🚀 Pre- and Post-Disaster Building Damage Assessment using Deep Learning
### Benchmark: ResNeXt-50 vs ConvNeXt-Tiny vs HybridSiam-BDA (Lite)
**Honors Project (Session 2026-2027)** | *Shri Ramdeobaba College of Engineering and Management, Nagpur*

This notebook trains and benchmarks three state-of-the-art architectures on the **Extended Building Damage (EBD) Dataset**:
1. **ResNeXt-50 Siamese UNet** (Grouped convolutions / Cardinality=32 baseline)
2. **ConvNeXt-Tiny Siamese UNet** (Modernized pure CNN baseline with 7x7 depthwise convolutions)
3. **HybridSiam-BDA (Lite)** (Novel Architecture: ConvNeXt-Tiny + Swin-V2-T + Multi-Scale Cross-Attention Difference Fusion + Attention Gates)

---
### 📊 About the EBD Dataset (14 GB):
- **Scale**: Over 18,000 bi-temporal image pairs (512x512 RGB) and 175,000+ building connected components.
- **Disasters**: 12 diverse natural disaster events (hurricanes, earthquakes, floods, volcanic eruptions, tornadoes).
- **Annotations**: Binary building localization (255: building, 0: background) and 4-level damage categories (1: no-damage, 2: minor damage, 3: major damage, 4: destroyed; 0: background).
- **Naming format**: `{disaster event}_{image ID}_{pre/post}_disaster`.

In [ ]:
# ── Cell 1: Check GPU Hardware ───────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 3: Install Required Dependencies ────────────────────────────────────
!pip install -q torch torchvision matplotlib opencv-python scikit-learn pillow albumentations shapely tqdm

In [ ]:
# ── Cell 4: Point Python at Project Directory ────────────────────────────────
import sys, os

PROJECT_DIR = '/content/drive/MyDrive/Honors_project'
assert os.path.isdir(PROJECT_DIR), f"Folder not found: {PROJECT_DIR} — check your Google Drive path!"

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Working directory :', os.getcwd())
print('Project path entry:', sys.path[0])

In [ ]:
# ── Cell 5: Smart EBD Dataset Setup (Skips unzipping if already done!) ────────
# Checks if dataset is already unzipped on fast local VM disk (/content/data).
# If yes, it completely SKIPS unzipping so re-running cells never repeats the unzip!
import os

LOCAL_DATA = '/content/data'
DRIVE_ZIP = '/content/drive/MyDrive/EBD_dataset.zip'
DRIVE_TAR = '/content/drive/MyDrive/EBD_dataset.tar.gz'
DRIVE_DIR = '/content/drive/MyDrive/EBD_dataset'

if os.path.exists(LOCAL_DATA) and len(os.listdir(LOCAL_DATA)) > 0:
    print(f'✅ Dataset already extracted in {LOCAL_DATA}! Skipping unzip step entirely.')
elif os.path.exists(DRIVE_ZIP):
    print(f'📦 Extracting {DRIVE_ZIP} to fast local disk {LOCAL_DATA} (one-time only)...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    !unzip -q -n "{DRIVE_ZIP}" -d "{LOCAL_DATA}"
    print('✅ Extraction complete!')
elif os.path.exists(DRIVE_TAR):
    print(f'📦 Extracting {DRIVE_TAR} to fast local disk {LOCAL_DATA} (one-time only)...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    !tar -xzf "{DRIVE_TAR}" -C "{LOCAL_DATA}"
    print('✅ Extraction complete!')
elif os.path.exists(DRIVE_DIR) and len(os.listdir(DRIVE_DIR)) > 0:
    print(f'✅ Found pre-extracted EBD dataset folder in Google Drive: {DRIVE_DIR}')
else:
    print('ℹ️ No archive or Drive folder found yet. Using synthetic EBD samples for verification.')

from config import ACTIVE_DATA_DIR
print(f'Active dataset path for training: {ACTIVE_DATA_DIR}')


In [ ]:
# ── Cell 6: Clear Stale Python Module Cache & Purge GPU VRAM ─────────────────
import sys, gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

stale = [m for m in sys.modules if any(
    k in m for k in ['train', 'config', 'dataset', 'models', 'losses', 'metrics', 'visualize']
)]
for m in stale:
    del sys.modules[m]

print(f'Cleared {len(stale)} cached module(s) and flushed GPU memory. Ready!')

In [ ]:
# ── Cell 7: Run 3-Way Comparative Experiment on EBD Dataset ──────────────────
# Trains ResNeXt-50, ConvNeXt-Tiny, and HybridSiam-BDA (Lite) sequentially
from train import run_comparative_experiment

results = run_comparative_experiment(epochs=30)

In [ ]:
# ── Cell 8: Visualize 3-Model Comparative Benchmark Chart ────────────────────
from visualize import plot_comparative_benchmark
from IPython.display import Image, display

chart_path = 'outputs/visualizations/comparative_bar.png'
plot_comparative_benchmark(
    results['ResNeXt50'],
    results['ConvNeXt'],
    results['HybridSiam'],
    save_path=chart_path
)

display(Image(chart_path))

In [ ]:
# ── Cell 9: Qualitative 6-Panel Damage Prediction Overlays (EBD) ─────────────
# Renders pre-image, post-image, ground truth footprints & damage, and model predictions
import os, torch
import numpy as np
from models.hybrid_siambda import HybridSiamBDA
from visualize import plot_bda_results
from dataset import EBDDataset
from config import ACTIVE_DATA_DIR, CHECKPOINT_DIR
from IPython.display import Image, display

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best checkpoint for HybridSiam-BDA (Lite)
ckpt_path = os.path.join(CHECKPOINT_DIR, 'HybridSiam_BDA_Lite_best.pth')
if os.path.exists(ckpt_path):
    model = HybridSiamBDA(pretrained=False).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    val_ds = EBDDataset(data_dir=ACTIVE_DATA_DIR, split='val', num_samples=20)
    pre_t, post_t, loc_gt, dam_gt = val_ds[0]

    with torch.no_grad():
        loc_pred, dam_pred = model(pre_t.unsqueeze(0).to(device), post_t.unsqueeze(0).to(device))
        pred_loc = (torch.sigmoid(loc_pred).squeeze().cpu().numpy() > 0.5).astype(np.float32)
        pred_dam = torch.argmax(dam_pred.squeeze(), dim=0).cpu().numpy()

    vis_save = 'outputs/visualizations/hybrid_siam_prediction.png'
    plot_bda_results(
        pre_t.permute(1, 2, 0).numpy(),
        post_t.permute(1, 2, 0).numpy(),
        loc_gt.squeeze().numpy(),
        dam_gt.numpy(),
        pred_loc,
        pred_dam,
        save_path=vis_save
    )
    display(Image(vis_save))
else:
    print('No checkpoint found yet. Run Cell 7 first!')